# Data quality analysis
This notebook does an analysis of the data quality of each dataset, and goes through measures to handle data quality issues such as duplicates, missing values, wrong formats etc.  

The notebook follows a structure where each dataset is analysed by first doing a qualitiative assesment, and then a quantitative assessment by checking the actual contents of data.

### General focus and aim
- Focus on these data:
    - Actual, departed trains (thus exclude cancelled trains)
    - Time period of analysis, limited by findings of data exploration: 01-08-2025 to 16-09-2026
- Aim to do these types of analytics:
    - Short analysis of departure delays compared to all departed trains
    - More thorough analysis of departure delays, isolated
        - Delay reason codes (Årsakskoder)
        - Also try: Which sub-processes stands for most delay?

### Quantitative analysis content:
- Data types/formats
    - For each relevant, chosen columns ^^
- Expected dates for analysis period
    - All dates covered?
    - Any gaps in the middle of the study period?
- Row level contents
    - Blank rows?
    - Duplicate rows?
    - Rows w/o train ID (Tognummer)?
    - Planned departures w/o actual departure times?
    - Actual departures w/o planned departure?
- Column level contents - missing values
    - For each relevant, chosen column ^^
    - (Distignuish between if missing values are ok or not, e.g. punctual departures have no delay code)
- Grain uniqueness and duplicates
    - LoggTogID
- Scope of edge cases
    - Overnight departures
    - Outliers
        - Very large delays
        - Very early departures (deviation from planned)

## Import of programming packages

In [27]:
import pandas as pd

## Terminallogg R25 and R26
*Combined R25 and R26 datasets as a result of the findings of the data exploration*

### Qualitative aspects

**Data collection procedure:** 
- A -- Content should contain all expected CargoNet train departures from Alnabru in the R25 and R26 calendar
- B -- Input data are mostly done manually in excel-sheets
- C -- Excel columns have strict data validation turned on to ensure proper formats
- D -- Some columns are calculated automatically using formulas (f.eks. the ones regarding 'Avvik' and 'LoggTogID')  

**Comment:**
- A -- the datasets works as a basis of which train departures to analyze, i.e. the datasets here are the ones to analyze for R25 and R26
- B & C -- The input is done manually and not automatically, and only the formatting is validated, not the actual content of the values. That makes the input values still vulnerable for wrong input values, typos, operator faults and/or routines for consistent/standardized inputs. This especially goes for timestamps, where one should do a quantitative assesment as well, to check the scope of data quality issues regarding the actual inputs. 
The 'Årsakskode'-column have more strict validation where users must choose from a list and can be considered to have a higher probability of "correct" input values, although the choice of code is done by the operator and thus dependent/vulnerable on their subjective interpretation.
- D -- The automatically calculated columns are dependent on input of other columns. These might still be used, but the formulas from here can easily be replicated and might rather be calculated again in this study for consistent calculations. 

**Relevant columns to use:**
- 'Merknad'
- 'Tognummer'
- 'Avgangsdato'
- 'Lastefrist'
- 'Ferdig opplastet'
- 'Frist lok'
- 'Lok ankommet'
- 'Planlagt klart fra terminal'
- 'Klart fra terminal'
- 'Planlagt avgang'
- 'Faktisk avgang'
- 'Årsakskode - avgangsforsinkelse'
- 'Kommentar'
- 'LoggTogID'

### Quantitative aspects

#### Preparation

In [28]:
# Import both datasets
df_terminallogg_r25 = pd.read_csv(
    "../data/Terminallogg R25.csv",
    sep=";",
    encoding="utf-8-sig"
)

df_terminallogg_r26 = pd.read_csv(
    "../data/Terminallogg R26.csv",
    sep=";",
    encoding="utf-8-sig"
)

In [29]:
# Define the inclusive date interval
start_date = pd.Timestamp("2025-08-01")
end_date = pd.Timestamp("2026-09-16")

# Convert Avgangsdato in both dataframes
for df in [df_terminallogg_r25, df_terminallogg_r26]:
    df["Avgangsdato"] = pd.to_datetime(
        df["Avgangsdato"],
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce"
    )

# Combine the datasets
df_terminallogg = pd.concat(
    [df_terminallogg_r25, df_terminallogg_r26],
    ignore_index=True
)

# Filter the combined dataset based on time period
df_terminallogg_filtered = (
    df_terminallogg[
        df_terminallogg["Avgangsdato"].between(
            start_date,
            end_date,
            inclusive="both"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

In [30]:
print("Number of rows:", len(df_terminallogg_filtered))
print(
    "Date range:",
    df_terminallogg_filtered["Avgangsdato"].min(),
    "to",
    df_terminallogg_filtered["Avgangsdato"].max()
)
print(df_terminallogg_filtered["Avgangsdato"].dtype)

Number of rows: 2340
Date range: 2025-08-01 00:00:00 to 2025-12-28 00:00:00
datetime64[us]


In [31]:
# Select the relevant columns for further analysis
relevant_columns = [
    "Merknad",
    "Tognummer",
    "Avgangsdato",
    "Lastefrist",
    "Ferdig opplastet",
    "Frist lok",
    "Lok ankommet",
    "Planlagt klart fra terminal",
    "Klart fra terminal",
    "Planlagt avgang",
    "Faktisk avgang",
    "Årsakskode - avgangsforsinkelse",
    "Kommentar",
    "LoggTogID"
]

df_terminallogg_filtered = (
    df_terminallogg_filtered[relevant_columns]
    .copy()
)

In [32]:
# Check filtering
print("Dimensions of filtered table:", df_terminallogg_filtered.shape)
df_terminallogg_filtered.columns.tolist()

Dimensions of filtered table: (2340, 14)


['Merknad',
 'Tognummer',
 'Avgangsdato',
 'Lastefrist',
 'Ferdig opplastet',
 'Frist lok',
 'Lok ankommet',
 'Planlagt klart fra terminal',
 'Klart fra terminal',
 'Planlagt avgang',
 'Faktisk avgang',
 'Årsakskode - avgangsforsinkelse',
 'Kommentar',
 'LoggTogID']

#### Actual analysis

##### Data types/formats

##### Expected dates for analysis period
- All dates covered?
- Any gaps in the middle of the study period?

##### Row level contents
- Blank rows?
- Duplicate rows?
- Rows w/o train ID (Tognummer)?
- Planned departures w/o actual departure times?
- Actual departures w/o planned departure?

##### Column level contents - missing values
- For each relevant, chosen column ^^
- (Distignuish between if missing values are ok or not, e.g. punctual departures have no delay code)

##### Grain uniqueness and duplicates

##### Scope of edge cases
- Overnight departures
- Outliers
    - Very large delays
    - Very early departures (deviation from planned)

### Summary and final verdict of this dataset

## ExcelReport-6f2dcb04-d1bc-4d7e-b997-cbd3a93fe1b3

### Qualitative aspects

**Data collection procedure:** 
- A -- 

**Comment:**


**Relevant columns to use:**
- A

### Quantitative aspects

#### Preparation

In [33]:
# # Define the inclusive date interval
# start_date = pd.Timestamp("2025-08-01")
# end_date = pd.Timestamp("2026-09-16")

In [34]:
# print("Number of rows:", len(df_terminallogg_filtered))
# print(
#     "Date range:",
#     df_terminallogg_filtered["Avgangsdato"].min(),
#     "to",
#     df_terminallogg_filtered["Avgangsdato"].max()
# )
# print(df_terminallogg_filtered["Avgangsdato"].dtype)

In [35]:
# # Select the relevant columns for further analysis
# relevant_columns = [
#     "Column name"
# ]

# df_terminallogg_filtered = (
#     df_terminallogg_filtered[relevant_columns]
#     .copy()
# )

In [36]:
# # Check filtering
# print("Dimensions of filtered table:", df_terminallogg_filtered.shape)
# df_terminallogg_filtered.columns.tolist()

#### Actual analysis

##### Data types/formats

##### Expected dates for analysis period
- All dates covered?
- Any gaps in the middle of the study period?

##### Row level contents
- Blank rows?
- Duplicate rows?
- Rows w/o train ID (Tognummer)?
- Planned departures w/o actual departure times?
- Actual departures w/o planned departure?

##### Column level contents - missing values
- For each relevant, chosen column ^^
- (Distignuish between if missing values are ok or not, e.g. punctual departures have no delay code)

##### Grain uniqueness and duplicates

##### Scope of edge cases
- Overnight departures
- Outliers
    - Very large delays
    - Very early departures (deviation from planned)

### Summary and final verdict of this dataset

## bronze_tp_operating_train_routes_ALB

### Qualitative aspects

**Data collection procedure:** 
- A -- 

**Comment:**


**Relevant columns to use:**
- A

### Quantitative aspects

#### Preparation

In [ ]:
# # Define the inclusive date interval
# start_date = pd.Timestamp("2025-08-01")
# end_date = pd.Timestamp("2026-09-16")

In [ ]:
# print("Number of rows:", len(df_terminallogg_filtered))
# print(
#     "Date range:",
#     df_terminallogg_filtered["Avgangsdato"].min(),
#     "to",
#     df_terminallogg_filtered["Avgangsdato"].max()
# )
# print(df_terminallogg_filtered["Avgangsdato"].dtype)

In [ ]:
# # Select the relevant columns for further analysis
# relevant_columns = [
#     "Column name"
# ]

# df_terminallogg_filtered = (
#     df_terminallogg_filtered[relevant_columns]
#     .copy()
# )

In [ ]:
# # Check filtering
# print("Dimensions of filtered table:", df_terminallogg_filtered.shape)
# df_terminallogg_filtered.columns.tolist()

#### Actual analysis

##### Data types/formats

##### Expected dates for analysis period
- All dates covered?
- Any gaps in the middle of the study period?

##### Row level contents
- Blank rows?
- Duplicate rows?
- Rows w/o train ID (Tognummer)?
- Planned departures w/o actual departure times?
- Actual departures w/o planned departure?

##### Column level contents - missing values
- For each relevant, chosen column ^^
- (Distignuish between if missing values are ok or not, e.g. punctual departures have no delay code)

##### Grain uniqueness and duplicates

##### Scope of edge cases
- Overnight departures
- Outliers
    - Very large delays
    - Very early departures (deviation from planned)

### Summary and final verdict of this dataset

## Årsakskodelister (IKKE REDIGER!

### Qualitative aspects

**Data collection procedure:** 
- A -- 

**Comment:**


**Relevant columns to use:**
- A

### Quantitative aspects

#### Preparation

In [ ]:
# # Define the inclusive date interval
# start_date = pd.Timestamp("2025-08-01")
# end_date = pd.Timestamp("2026-09-16")

In [ ]:
# print("Number of rows:", len(df_terminallogg_filtered))
# print(
#     "Date range:",
#     df_terminallogg_filtered["Avgangsdato"].min(),
#     "to",
#     df_terminallogg_filtered["Avgangsdato"].max()
# )
# print(df_terminallogg_filtered["Avgangsdato"].dtype)

In [ ]:
# # Select the relevant columns for further analysis
# relevant_columns = [
#     "Column name"
# ]

# df_terminallogg_filtered = (
#     df_terminallogg_filtered[relevant_columns]
#     .copy()
# )

In [ ]:
# # Check filtering
# print("Dimensions of filtered table:", df_terminallogg_filtered.shape)
# df_terminallogg_filtered.columns.tolist()

#### Actual analysis

##### Data types/formats

##### Expected dates for analysis period
- All dates covered?
- Any gaps in the middle of the study period?

##### Row level contents
- Blank rows?
- Duplicate rows?
- Rows w/o train ID (Tognummer)?
- Planned departures w/o actual departure times?
- Actual departures w/o planned departure?

##### Column level contents - missing values
- For each relevant, chosen column ^^
- (Distignuish between if missing values are ok or not, e.g. punctual departures have no delay code)

##### Grain uniqueness and duplicates

##### Scope of edge cases
- Overnight departures
- Outliers
    - Very large delays
    - Very early departures (deviation from planned)

### Summary and final verdict of this dataset